# 11 · Input, output and private state schemas

**Goal:** by the end you can decide exactly which keys a graph accepts and which it returns, while its nodes keep
working with extra keys that stay hidden inside.

## The idea in plain English

Until now `invoke` returned **everything**: `expression`, `a`, `op`, `b`, `result` and `answer`. Someone using our
calculator only cares about the `answer`. The other keys are the graph's internal workings.

LangGraph lets you give a graph three schemas:

- an **input schema**: the keys `invoke` accepts (here just `expression`),
- an **output schema**: the keys `invoke` returns (here just `answer`),
- the **overall state**: every key the nodes use, including the internal ones.

You pass them as `StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)`. Keys used only
inside the graph are called **private**: they're never accepted from outside and never returned.

**Analogy:** a restaurant. You give your order (input) and get your dish (output). The kitchen's notes, timers and
half-chopped vegetables (private state) stay in the kitchen.

## Picture

```mermaid
flowchart LR
    I["input<br/>expression"] --> parse
    subgraph graph["inside the graph (overall state)"]
        parse --> calculate --> format
        notes["a, op, b, result<br/>(private)"]
    end
    format --> O["output<br/>answer"]
```

## Step 1: Three schemas

A `TypedDict` can **inherit** from others, which combines their keys. `OverallState(InputState, OutputState)` starts
with `expression` and `answer`, then adds the internal keys. That way the three schemas can't drift apart.

In [1]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class InputState(TypedDict):
    expression: str


class OutputState(TypedDict):
    answer: str


class OverallState(InputState, OutputState):
    a: float
    op: str
    b: float
    result: float

Printing each schema's keys shows how inheritance combined them: `OverallState` has all six keys, the two public ones first.

In [2]:
print("input keys:  ", list(InputState.__annotations__))
print("output keys: ", list(OutputState.__annotations__))
print("overall keys:", list(OverallState.__annotations__))

input keys:   ['expression']
output keys:  ['answer']
overall keys: ['expression', 'answer', 'a', 'op', 'b', 'result']


## Step 2: The nodes

The pipeline from lesson 04. Every node works with `OverallState`, so it can read and write the internal keys.
`parse` prints what it receives, which will be useful in Step 5.

In [3]:
def parse(state: OverallState) -> dict:
    print("  parse received:", state)
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}

`calculate` does the arithmetic and `format_answer` builds the answer. Both use `OverallState`, so they can read the internal keys `a`, `op`, `b` and `result`.

In [4]:
def calculate(state: OverallState) -> dict:
    a, op, b = state["a"], state["op"], state["b"]
    if op == "+":
        return {"result": a + b}
    elif op == "-":
        return {"result": a - b}
    elif op == "*":
        return {"result": a * b}
    return {"result": a / b}


def format_answer(state: OverallState) -> dict:
    return {"answer": f"{state['expression']} = {state['result']:g}"}

## Step 3: Build it with and without schemas

To compare, `build_pipeline` builds the same graph twice. The only difference is the two schema arguments. Their
default, `None`, means "no separate schema: use the overall state", which is what we've had so far.

In [5]:
def build_pipeline(input_schema=None, output_schema=None):
    builder = StateGraph(OverallState, input_schema=input_schema, output_schema=output_schema)
    builder.add_node("parse", parse)
    builder.add_node("calculate", calculate)
    builder.add_node("format", format_answer)
    builder.add_edge(START, "parse")
    builder.add_edge("parse", "calculate")
    builder.add_edge("calculate", "format")
    builder.add_edge("format", END)
    return builder.compile()


everything = build_pipeline()
calculator = build_pipeline(input_schema=InputState, output_schema=OutputState)
print(calculator.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	calculate(calculate)
	format(format)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	calculate --> format;
	parse --> calculate;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 4: Before and after

Without schemas, `invoke` returns all six keys. With the output schema, it returns only `answer`. The nodes did
exactly the same work in both runs. The output schema only filters what comes back.

In [6]:
print("without schemas:", everything.invoke({"expression": "3 + 4"}))
print()
print("with schemas:   ", calculator.invoke({"expression": "3 + 4"}))

  parse received: {'expression': '3 + 4'}
without schemas: {'expression': '3 + 4', 'answer': '3 + 4 = 7', 'a': 3.0, 'op': '+', 'b': 4.0, 'result': 7.0}

  parse received: {'expression': '3 + 4'}
with schemas:    {'answer': '3 + 4 = 7'}


## Step 5: The input schema filters what comes in

Here a caller tries to sneak in `a = 100`. Because `a` isn't in `InputState`, LangGraph drops it, and `parse`
receives only `expression`. That stops callers from setting internal values they shouldn't touch.

In [7]:
print(calculator.invoke({"expression": "3 + 4", "a": 100.0}))

  parse received: {'expression': '3 + 4'}
{'answer': '3 + 4 = 7'}


## Step 6: The schemas describe the graph to others

A compiled graph can describe its own input and output as a **JSON Schema**, a standard format other programs (for
example a web API) use to check data. Only the public keys appear.

In [8]:
print("input: ", calculator.get_input_jsonschema())
print("output:", calculator.get_output_jsonschema())

input:  {'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'InputState', 'type': 'object'}
output: {'properties': {'answer': {'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'OutputState', 'type': 'object'}


## Step 7: Private state between two nodes

Sometimes two nodes need to pass something between them that no other node cares about. Here a new `tokenize` node
splits the text into pieces (`tokens`), and `parse` reads them. `tokens` lives in its own small schema,
`TokenState`, and isn't in `OverallState` at all.

The return type `-> TokenState` on `tokenize` tells LangGraph about the extra key. The type hint `state: TokenState`
on `parse_tokens` means it **receives only** the keys in `TokenState`.

In [9]:
class TokenState(TypedDict):
    tokens: list[str]


def tokenize(state: InputState) -> TokenState:
    return {"tokens": state["expression"].split()}


def parse_tokens(state: TokenState) -> dict:
    print("  parse_tokens received:", state)
    left, op, right = state["tokens"]
    return {"a": float(left), "op": op, "b": float(right)}

The pipeline gets one more step at the front: `START → tokenize → parse → calculate → format → END`. The input and output schemas are the same as before.

In [10]:
builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
builder.add_node("tokenize", tokenize)
builder.add_node("parse", parse_tokens)
builder.add_node("calculate", calculate)
builder.add_node("format", format_answer)
builder.add_edge(START, "tokenize")
builder.add_edge("tokenize", "parse")
builder.add_edge("parse", "calculate")
builder.add_edge("calculate", "format")
builder.add_edge("format", END)
with_tokens = builder.compile()

print(with_tokens.invoke({"expression": "6 * 7"}))

  parse_tokens received: {'tokens': ['6', '*', '7']}
{'answer': '6 * 7 = 42'}


`parse_tokens` received only `{'tokens': [...]}`: not `expression`, and not the other keys. And `tokens` isn't in the
result. It was private to the two nodes that used it.

This is the same rule you met in lesson 06: **LangGraph reads a node's type hint to decide which keys it sees.**
Usually every node uses the overall state, so it sees everything. A narrower hint gives a narrower view.

## What just happened

- **Step 1** made three schemas. `OverallState` inherited `expression` and `answer` and added `a`, `op`, `b`,
  `result`.
- **Step 4** ran the same nodes twice. Without schemas the result had six keys. With `output_schema=OutputState` it
  was just `{'answer': '3 + 4 = 7'}`.
- **Step 5** passed an extra `a: 100.0`. `parse` received only `{'expression': '3 + 4'}`, and the answer was still
  `3 + 4 = 7`.
- **Step 6** printed the JSON Schemas: the input requires `expression`, and the output has `answer`.
- **Step 7** passed `tokens` privately from `tokenize` to `parse_tokens`, which received only `{'tokens': ['6', '*',
  '7']}`. The output was still only `answer`.

## Common mistakes

1. **Using the old argument names from older tutorials.** `StateGraph(OverallState, input=InputState)` still works
   for now, but warns:
   `LangGraphDeprecatedSinceV05: `input` is deprecated and will be removed. Please use `input_schema` instead.`
   Fix: use `input_schema=` and `output_schema=` (and `context_schema=` in lesson 12).
2. **Reading an internal key from the result.** With `output_schema=OutputState`, `result["result"]` fails:
   `KeyError: 'result'`.
   Fix: add the key to the output schema if callers need it.
3. **Expecting the input schema to check the input.** Leaving out `expression` isn't caught at the door. `parse`
   fails later with `KeyError: 'expression'`. Like `TypedDict` in lesson 02, the input schema only filters keys; it
   doesn't validate them.
   Fix: check required input in your first node, or use a Pydantic model (lesson 02) if you need validation.

## Try it

1. Change the output so callers get both the `answer` and the numeric `result`. What do you need to change, and what
   does `invoke({"expression": "6 * 7"})` return?
2. Add a `request_id: str` that the caller passes in and gets back unchanged, so they can match answers to
   requests. Which schemas need it? (No node needs to touch it.)

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Input schema**: the keys a graph accepts in `invoke`; set with `input_schema=`. Other keys are dropped.
- **Output schema**: the keys a graph returns from `invoke`; set with `output_schema=`.
- **Overall state**: the full state schema the nodes work with, given as the first argument of `StateGraph`.
- **Private state**: keys used inside the graph that are never accepted from outside or returned.
- **JSON Schema**: a standard description of data, which `get_input_jsonschema()` and `get_output_jsonschema()`
  return.